# Hill Climbing

## Exercise 3: Reach the highest summit

### Intermediate

In [9]:
import math
import random             	# just for generating random mountains

# generate random mountains
w = [random.random()/3, random.random()/3, random.random()/3]
h = [1.+math.sin(1+x/6.)*w[0]+math.sin(-.3+x/9.)*w[1]+math.sin(-.2+x/30.)*w[2] for x in range(100)]
h[0] = 0.0; h[99] = 0.0

def climb(x, h):
    # keep climbing until we've found a summit
    summit = False

    # edit here
    while not summit:
        summit = True         # stop unless there's a way up
        if h[x + 1] > h[x]:
            x = x + 1         # right is higher, go there
            summit = False    # and keep going
        elif h[x - 1] > h[x]:
            x = x -1
            summit = False
    return x


def main(h):

    # start at a random place
    x0 = random.randint(1, 98)
    x = climb(x0, h)

    print("Venla started at %d and got to %d" % (x0, x))
    return x0, x

main(h)

(61, 77)

### Advanced

In [10]:
import math
import random             	# just for generating random mountains

# generate random mountains

w = [.05, random.random()/3, random.random()/3]
h = [1.+math.sin(1+x/.6)*w[0]+math.sin(-.3+x/9.)*w[1]+math.sin(-.2+x/30.)*w[2] for x in range(100)]

def climb(x, h):
    # keep climbing until we've found a summit
    summit = False

    # edit here
    while not summit:
        summit = True         # stop unless there's a way up

        best_x = x
        best_h = h[x]

        for step in range(1,6):
            if x + step <= 99 and h[x + step] > best_h:
                best_h = h[x + step]
                best_x = x + step

            if x - step >= 0 and h[x - step] > best_h:
                best_h = h[x - step]
                best_x = x - step

        if best_x != x:
            x = best_x
            summit = False

    return x


def main(h):
    # start at a random place
    x0 = random.randint(1, 98)
    x = climb(x0, h)

    return x0, x

main(h)

(47, 72)

## Exercise 4: Probabilities

### Intermediate

In [11]:
import random

def main():
    prob = 0.80
    if random.random() < prob:
        print('dog')
    else:
        print("cat")

main()

dog


### Advanced

In [12]:
import random

def main():
    x = random.random()

    if x < 0.1:
        favourite = "bats"
    elif x < 0.2:
        favourite = "cats"
    else:
        favourite = "dogs"

    print("I love " + favourite)
main()

I love dogs


## Exercise 5: Warm-up Temperature

### Advanced

In [13]:
import random
import math


def accept_prob(S_old, S_new, T):
    # this is the acceptance "probability" in the greedy hill-climbing method
    # where new solutions are accepted if and only if they are better
    # than the old one.
    # change it to be the acceptance probability in simulated annealing

    if S_new > S_old:
        return 1.0
    else:
        return math.exp((S_new - S_old)/T)


# the above function will be used as follows. this is shown just for
# your information; you don't have to change anything here
def accept(S_old, S_new, T):
    if random.random() < accept_prob(S_old, S_new, T):
        print(True)
    else:
        print(False)

## Exercise 6: Simulated Annealing

### Intermediate

In [14]:
import math, random        	# just for generating random mountains
import numpy as np

n = 10000 # size of the problem: number of possible solutions x = 0, ..., n-1

# generate random mountains
def mountains(n):
    h = [0]*n
    for i in range(50):
        c = random.randint(20, n-20)
        w = random.randint(3, int(math.sqrt(n/5)))**2
        s = random.random()
        h[max(0, c-w):min(n, c+w)] = [h[i] + s*(w-abs(c-i)) for i in range(max(0, c-w), min(n, c+w))]

    # scale the height so that the lowest point is 0.0 and the highest peak is 1.0
    low = min(h)
    high = max(h)
    h = [y - low for y in h]
    h = [y / (high-low) for y in h]
    return h

h = mountains(n)

# start at a random place
x0 = random.randint(1, n-1)
x = x0

# keep climbing for 5000 steps
steps = 5000

def main(h, x):
    n = len(h)
    # the climbing starts here
    for step in range(steps):
        # this is our temperature to to be used for simulated annealing
        # it starts large and decreases with each step. you don't have to change this
        T = 2*max(0, ((steps-step*1.2)/steps))**3

        # let's try randomly moving (max. 1000 steps) left or right
        # making sure we don't fall off the edge of the world at 0 or n-1
        # the height at this point will be our candidate score, S_new
        # while the height at our current location will be S_old
        x_new = random.randint(max(0, x-1000), min(n-1, x+1000))

        if h[x_new] > h[x]:
            x = x_new           # the new position is higher, go there
        else:
            if T > 0:
                S_old = h[x]
                S_new = h[x_new]
                prob = math.exp(-(S_old - S_new) / T)
                if random.random() < prob:
                    x = x_new    # accept the worse solution anyway
            # if T == 0, never accept a worse solution (prob would be 0 anyway)

    return x

x = main(h, x0)
print("ended up at %d, highest point is %d" % (x, np.argmax(h)))


ended up at 8439, highest point is 8438


### Advanced

In [15]:
import numpy as np
import random

N = 100     # size of the problem is N x N
steps = 3000    # total number of iterations
tracks = 50

# generate a landscape with multiple local optima
def generator(x, y, x0=0.0, y0=0.0):
    return np.sin((x/N-x0)*np.pi)+np.sin((y/N-y0)*np.pi)+\
        .07*np.cos(12*(x/N-x0)*np.pi)+.07*np.cos(12*(y/N-y0)*np.pi)

x0 = np.random.random() - 0.5
y0 = np.random.random() - 0.5
h = np.fromfunction(np.vectorize(generator), (N, N), x0=x0, y0=y0, dtype=int)
peak_x, peak_y = np.unravel_index(np.argmax(h), h.shape)

# starting points
x = np.random.randint(0, N, tracks)
y = np.random.randint(0, N, tracks)

def main():
    global x
    global y

    for step in range(steps):
        # add a temperature schedule here
        T = 2*max(0, ((steps-step*1.2)/steps))**3
        # update solutions on each search track
        for i in range(tracks):
            # try a new solution near the current one
            x_new = np.random.randint(max(0, x[i]-2), min(N, x[i]+2+1))
            y_new = np.random.randint(max(0, y[i]-2), min(N, y[i]+2+1))
            S_old = h[x[i], y[i]]
            S_new = h[x_new, y_new]

            # change this to use simulated annealing
            if S_new > S_old:
                x[i], y[i] = x_new, y_new   # new solution is better, go there
            else:
                if T > 0:
                    prob = np.exp(-(S_old - S_new) / T)
                    if random.random() < prob:
                        x[i], y[i] = x_new, y_new   # accept the worse solution anyway
                # if T == 0, never accept a worse solution

    # Number of tracks found the peak
    print(sum([x[j] == peak_x and y[j] == peak_y for j in range(tracks)]))
main()


39
